## Градиентный спуск

В этом задании мы реализуем алгоритм градиентного спуска и исследуем его поведение на функции с несколькими локальными минимумами.

Это поможет получить интуицию о том, как работает обучение нейронных сетей: нейросеть оптимизирует функцию потерь (loss), и градиентный спуск — основной инструмент для этого.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from matplotlib import cm

---
### 1. Функция и её градиент

Рассмотрим функцию двух переменных:

$$f(x, y) = x^4 + y^4 - 2x^2 - 2y^2 - x - 0.5y$$

Эта функция имеет **4 локальных минимума** с разной глубиной (один из них — глобальный). Это делает её интересной для исследования: в зависимости от начальной точки, градиентный спуск может прийти в разные минимумы.

Градиент (вектор частных производных):

$$\nabla f(x, y) = \begin{pmatrix} 4x^3 - 4x - 1 \\ 4y^3 - 4y - 0.5 \end{pmatrix}$$

#### 1.1. Реализуйте функцию `f(X)` в векторизованном виде

Функция должна принимать массив `X` формы `(n, 2)`, где каждая строка — точка $(x, y)$, и возвращать вектор значений формы `(n,)`.

In [ ]:
def f(X: np.ndarray) -> np.ndarray:
    """X: (n, 2) -> (n,)"""
    x, y = X[:, 0], X[:, 1]
    return ...

# проверка
_test_X = np.array([[0.0, 0.0], [1.0, 1.0], [-1.0, -1.0]])
_test_expected = np.array([0.0, -3.5, -0.5])
assert np.allclose(f(_test_X), _test_expected), f"{f(_test_X)} != {_test_expected}"

#### 1.2. Реализуйте функцию `grad(X)` для вычисления градиента

Функция должна принимать `X` формы `(n, 2)` и возвращать массив градиентов той же формы `(n, 2)`.

Формулы:
- $\frac{\partial f}{\partial x} = 4x^3 - 4x - 1$
- $\frac{\partial f}{\partial y} = 4y^3 - 4y - 0.5$

In [ ]:
def grad(X: np.ndarray) -> np.ndarray:
    """X: (n, 2) -> (n, 2)"""
    x, y = X[:, 0], X[:, 1]
    df_dx = ...
    df_dy = ...
    return np.stack([df_dx, df_dy], axis=1)

# проверка: в точке (0, 0) градиент = (-1, -0.5)
_test_grad = grad(np.array([[0.0, 0.0]]))
assert np.allclose(_test_grad, [[-1.0, -0.5]]), f"{_test_grad} != [[-1.0, -0.5]]"

---
### 2. Визуализация функции

Прежде чем запускать оптимизацию, давайте посмотрим на ландшафт функции.

#### 2.1. Контурный график (matplotlib)

In [ ]:
# создаём сетку точек для визуализации
grid_x = np.linspace(-2, 2, 200)
grid_y = np.linspace(-2, 2, 200)
XX, YY = np.meshgrid(grid_x, grid_y)
# формируем массив точек (n, 2) для подачи в функцию f
grid_points = np.stack([XX.ravel(), YY.ravel()], axis=1)
ZZ = f(grid_points).reshape(XX.shape)

fig, ax = plt.subplots(figsize=(8, 6))
contour = ax.contourf(XX, YY, ZZ, levels=50, cmap='RdYlBu_r')
ax.contour(XX, YY, ZZ, levels=20, colors='k', linewidths=0.3, alpha=0.5)
fig.colorbar(contour, ax=ax, label='f(x, y)')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Контурный график f(x, y)')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

#### 2.2. 3D-график (matplotlib)

In [ ]:
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(XX, YY, ZZ, cmap='RdYlBu_r', alpha=0.8, edgecolor='none')
fig.colorbar(surf, ax=ax, shrink=0.5, label='f(x, y)')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('f(x, y)')
ax.set_title('Поверхность f(x, y)')
ax.view_init(elev=25, azim=-60)
plt.tight_layout()
plt.show()

#### 2.3. 3D-график (plotly)

Интерактивный график — можно вращать и приближать мышкой.

In [ ]:
fig = go.Figure(data=[go.Surface(
    x=grid_x, y=grid_y, z=ZZ,
    colorscale='RdYlBu_r', opacity=0.9,
    contours_z=dict(show=True, usecolormap=True, highlightcolor="limegreen", project_z=True)
)])
fig.update_layout(
    title='Поверхность f(x, y)',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='f(x, y)'),
    width=800, height=600
)
fig.show()

#### 2.4. Визуализация градиента

Посмотрим на поле градиентов — стрелки указывают направление наискорейшего *возрастания* функции. Градиентный спуск двигается в противоположном направлении (анти-градиент).

In [ ]:
# разреженная сетка для стрелок
q_x = np.linspace(-2, 2, 15)
q_y = np.linspace(-2, 2, 15)
QX, QY = np.meshgrid(q_x, q_y)
q_points = np.stack([QX.ravel(), QY.ravel()], axis=1)
G = grad(q_points)
# нормализуем стрелки к единичной длине, чтобы показать только направление
G_norm = np.linalg.norm(G, axis=1, keepdims=True)
G_unit = np.where(G_norm > 1e-8, G / G_norm, 0)
GX = G_unit[:, 0].reshape(QX.shape)
GY = G_unit[:, 1].reshape(QY.shape)

fig, ax = plt.subplots(figsize=(8, 6))
contour = ax.contourf(XX, YY, ZZ, levels=50, cmap='RdYlBu_r')
# рисуем анти-градиент (направление спуска)
ax.quiver(QX, QY, -GX, -GY, color='black', alpha=0.6)
fig.colorbar(contour, ax=ax, label='f(x, y)')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Поле анти-градиента (направление спуска)')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

#### 2.5. Норма градиента в 3D

Норма (длина) вектора градиента показывает, насколько "крутой" склон в каждой точке. В минимумах норма градиента близка к нулю.

In [ ]:
G_full = grad(grid_points)
grad_norm = np.linalg.norm(G_full, axis=1).reshape(XX.shape)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(XX, YY, grad_norm, cmap='magma_r', alpha=0.8, edgecolor='none')
fig.colorbar(surf, ax=ax, shrink=0.5, label='||∇f(x, y)||')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('||∇f||')
ax.set_title('Норма градиента ||∇f(x, y)||')
ax.view_init(elev=25, azim=-60)
plt.tight_layout()
plt.show()

#### 2.6. Норма градиента в 3D (plotly)

In [ ]:
fig = go.Figure(data=[go.Surface(
    x=grid_x, y=grid_y, z=grad_norm,
    colorscale='Magma_r', opacity=0.9,
)])
fig.update_layout(
    title='Норма градиента ||∇f(x, y)||',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='||∇f||'),
    width=800, height=600
)
fig.show()

---
### 3. Градиентный спуск

Алгоритм градиентного спуска:

$$\mathbf{x}_{t+1} = \mathbf{x}_t - \eta \nabla f(\mathbf{x}_t)$$

где $\eta$ — скорость обучения (learning rate).

#### 3.1. Реализуйте градиентный спуск

Напишите функцию, которая:
- Принимает начальные точки `X0` формы `(n, 2)`, learning rate `lr` и число шагов `n_steps`
- Возвращает историю всех позиций формы `(n_steps + 1, n, 2)` (включая начальную точку)

Заметьте, что благодаря векторизации функций `f` и `grad` мы можем запускать градиентный спуск из нескольких начальных точек одновременно!

In [ ]:
def gradient_descent(X0: np.ndarray, lr: float, n_steps: int) -> np.ndarray:
    """
    X0: (n, 2) — начальные точки
    lr: скорость обучения
    n_steps: количество шагов
    Возвращает: (n_steps + 1, n, 2) — история позиций
    """
    history = [X0.copy()]
    X = X0.copy()
    for _ in range(n_steps):
        ...  # один шаг градиентного спуска
        history.append(X.copy())
    return np.array(history)

#### 3.2. Запустите из нескольких начальных точек и визуализируйте траектории

Попробуйте `lr=0.01` и `n_steps=300`. Точки разбросаны по разным "бассейнам" — посмотрите, в какие минимумы они приходят.

In [ ]:
# начальные точки
X0 = np.array([
    [-1.5, -1.5],
    [-1.5,  1.5],
    [ 1.5, -1.5],
    [ 1.5,  1.5],
    [ 0.0,  0.0],
    [-0.5,  0.5],
])

history = gradient_descent(X0, lr=0.01, n_steps=300)
print("Форма history:", history.shape)  # должно быть (301, 6, 2)

# конечные точки
print("\nКонечные точки:")
for i in range(len(X0)):
    x_end = history[-1, i]
    print(f"  Старт {X0[i]} -> Финиш ({x_end[0]:.3f}, {x_end[1]:.3f}), f = {f(x_end[np.newaxis])[0]:.4f}")

In [ ]:
# визуализация траекторий на контурном графике
fig, ax = plt.subplots(figsize=(8, 6))
contour = ax.contourf(XX, YY, ZZ, levels=50, cmap='RdYlBu_r')
ax.contour(XX, YY, ZZ, levels=20, colors='k', linewidths=0.3, alpha=0.5)

colors = plt.cm.Set1(np.linspace(0, 1, len(X0)))
for i in range(len(X0)):
    traj = history[:, i, :]  # (n_steps+1, 2)
    ax.plot(traj[:, 0], traj[:, 1], '-', color=colors[i], linewidth=1.5, alpha=0.8)
    ax.plot(traj[0, 0], traj[0, 1], 'o', color=colors[i], markersize=8)
    ax.plot(traj[-1, 0], traj[-1, 1], '*', color=colors[i], markersize=15)

fig.colorbar(contour, ax=ax, label='f(x, y)')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Траектории градиентного спуска (lr=0.01)')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

---
### 4. Влияние learning rate

Скорость обучения (learning rate) — один из важнейших гиперпараметров при обучении нейросетей.

#### 4.1. Эксперимент

Запустите градиентный спуск из одной и той же точки `[0.0, 0.0]` с тремя разными learning rate: `0.001`, `0.01`, `0.25`. Используйте `n_steps=200`.

Визуализируйте:
1. Траектории на контурном графике
2. Значение функции $f$ на каждом шаге (кривая сходимости)

In [ ]:
learning_rates = [0.001, 0.01, 0.25]
X0_single = np.array([[0.0, 0.0]])

histories = {}
for lr in learning_rates:
    histories[lr] = gradient_descent(X0_single, lr=lr, n_steps=200)

In [ ]:
# траектории
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, lr in zip(axes, learning_rates):
    ax.contourf(XX, YY, ZZ, levels=50, cmap='RdYlBu_r')
    ax.contour(XX, YY, ZZ, levels=20, colors='k', linewidths=0.3, alpha=0.5)
    traj = histories[lr][:, 0, :]
    ax.plot(traj[:, 0], traj[:, 1], 'k-', linewidth=1.5)
    ax.plot(traj[0, 0], traj[0, 1], 'go', markersize=10, label='старт')
    ax.plot(traj[-1, 0], traj[-1, 1], 'r*', markersize=15, label='финиш')
    ax.set_title(f'lr = {lr}')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_aspect('equal')
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# кривые сходимости
fig, ax = plt.subplots(figsize=(8, 5))
for lr in learning_rates:
    traj = histories[lr][:, 0, :]
    values = f(traj)
    ax.plot(values, label=f'lr = {lr}')
ax.set_xlabel('Шаг')
ax.set_ylabel('f(x, y)')
ax.set_title('Сходимость: значение функции по шагам')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Что вы наблюдаете?
- Слишком маленький lr → медленная сходимость, не успевает дойти до минимума
- Подходящий lr → быстрая и стабильная сходимость к глобальному минимуму
- Слишком большой lr → нестабильность, «прыжки» мимо минимума, сходимость к неоптимальному решению

---
### 5. Бонус: градиентный спуск с инерцией (momentum)

В реальных задачах обучения нейросетей направление градиента может быть "шумным", и обычный градиентный спуск может медленно сходиться. Один из популярных приёмов — добавление *инерции* (momentum), которая "сглаживает" траекторию:

$$
\begin{align*}
\mathbf{g}_0 &= 0 \\
\mathbf{g}_t &= - \eta \nabla f(\mathbf{x}_{t-1}) + \lambda \mathbf{g}_{t-1} \\
\mathbf{x}_{t} &= \mathbf{x}_{t-1} + \mathbf{g}_t
\end{align*}
$$

где $\lambda$ — коэффициент инерции (обычно 0.9).

Реализуйте градиентный спуск с инерцией и сравните с обычным.

In [ ]:
def gradient_descent_momentum(
    X0: np.ndarray, lr: float, momentum: float, n_steps: int
) -> np.ndarray:
    """
    X0: (n, 2) — начальные точки
    lr: скорость обучения
    momentum: коэффициент инерции (lambda)
    n_steps: количество шагов
    Возвращает: (n_steps + 1, n, 2) — история позиций
    """
    history = [X0.copy()]
    X = X0.copy()
    g = np.zeros_like(X)  # начальная инерция = 0
    for _ in range(n_steps):
        ...  # один шаг с инерцией
        history.append(X.copy())
    return np.array(history)

In [ ]:
# сравнение: обычный GD vs GD с momentum
# используем маленький lr, чтобы увидеть ускорение от momentum
X0_cmp = np.array([[0.0, 0.0]])
lr_cmp = 0.003
n_steps_cmp = 200

hist_vanilla = gradient_descent(X0_cmp, lr=lr_cmp, n_steps=n_steps_cmp)
hist_momentum = gradient_descent_momentum(X0_cmp, lr=lr_cmp, momentum=0.9, n_steps=n_steps_cmp)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, hist, title in zip(axes,
                            [hist_vanilla, hist_momentum],
                            ['Без momentum', 'С momentum (λ=0.9)']):
    ax.contourf(XX, YY, ZZ, levels=50, cmap='RdYlBu_r')
    ax.contour(XX, YY, ZZ, levels=20, colors='k', linewidths=0.3, alpha=0.5)
    traj = hist[:, 0, :]
    ax.plot(traj[:, 0], traj[:, 1], 'k-', linewidth=1.5)
    ax.plot(traj[0, 0], traj[0, 1], 'go', markersize=10)
    ax.plot(traj[-1, 0], traj[-1, 1], 'r*', markersize=15)
    ax.set_title(title)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

In [ ]:
# сравнение кривых сходимости
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# значение функции
ax = axes[0]
for hist, label in [(hist_vanilla, 'Vanilla GD'), (hist_momentum, 'GD + Momentum')]:
    vals = f(hist[:, 0, :])
    ax.plot(vals, label=label)
ax.set_xlabel('Шаг')
ax.set_ylabel('f(x, y)')
ax.set_title('Значение функции')
ax.legend()
ax.grid(True, alpha=0.3)

# норма градиента
ax = axes[1]
for hist, label in [(hist_vanilla, 'Vanilla GD'), (hist_momentum, 'GD + Momentum')]:
    grads = grad(hist[:, 0, :])
    norms = np.linalg.norm(grads, axis=1)
    ax.plot(norms, label=label)
ax.set_xlabel('Шаг')
ax.set_ylabel('||∇f||')
ax.set_title('Норма градиента')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()